In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, confusion_matrix)

def main():

    print("PARTE A: ENTENDER EL PROBLEMA")
    print("1. ¿Es regresión o clasificación? ¿Por qué?")
    print("   -> Es Clasificación Binaria, porque la variable objetivo ('cancelo') tiene dos categorías posibles: 'Sí' (1) o 'No' (0).")
    print("\n2. Identifica X y Y:")
    print("   -> Y (Target): 'cancelo'")
    print("   -> X (Features): 'meses_activo', 'visitas_mes', 'quejas', 'plan_mensual'")
    print("\n3. Pregunta de reflexión: ¿Usar 'recibió encuesta de salida' como X?")
    print("   -> No. Introduciría 'Fuga de Datos' (Data Leakage) o información del futuro, ya que la encuesta se aplica DESPUÉS de que el socio cancela.\n")

    print("PARTE B: PREPARAR LOS DATOS")
    # Step 1: Cargar datos
    df = pd.read_csv('gimnasio_cancelaciones.csv')

    print("Conteo inicial de cancelaciones:")
    print(df['cancelo'].value_counts())
    print()

    # Step 2: Convertir texto a números
    df['plan_mensual'] = df['plan'].apply(lambda x: 1 if str(x).strip() == 'Mensual' else 0)
    df['cancelo'] = df['cancelo'].apply(lambda x: 1 if str(x).strip() in ['Sí', 'Si'] else 0)

    # Step 3: Separar X e Y
    X = df[['meses_activo', 'visitas_mes', 'quejas', 'plan_mensual']]
    y = df['cancelo']

    # Step 4: Apartar datos de prueba (70% Train, 30% Test con estratificación)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    print(f"Muestras de entrenamiento: {X_train.shape[0]}")
    print(f"Muestras de prueba: {X_test.shape[0]}\n")

    print("PARTE C: ENTRENAR Y EVALUAR TRES MODELOS")
    # 1. Regresión Logística
    modelo_lr = LogisticRegression(random_state=42)
    modelo_lr.fit(X_train, y_train)
    y_pred_lr = modelo_lr.predict(X_test)

    # 2. Árbol sin límite
    modelo_dt_unlimited = DecisionTreeClassifier(random_state=42)
    modelo_dt_unlimited.fit(X_train, y_train)
    y_pred_dt_unlimited = modelo_dt_unlimited.predict(X_test)

    # 3. Árbol limitado (max_depth=3)
    modelo_dt_limited = DecisionTreeClassifier(max_depth=3, random_state=42)
    modelo_dt_limited.fit(X_train, y_train)
    y_pred_dt_limited = modelo_dt_limited.predict(X_test)

    # Función auxiliar para imprimir métricas
    def evaluar(nombre, y_true, y_pred):
        print(f"--- {nombre} ---")
        print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
        print(f"Precision: {precision_score(y_true, y_pred):.4f}")
        print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
        print("Matriz de Confusión:")
        print(confusion_matrix(y_true, y_pred))
        print()

    evaluar("Regresión Logística", y_test, y_pred_lr)
    evaluar("Árbol sin Límite", y_test, y_pred_dt_unlimited)
    evaluar("Árbol Limitado (max_depth=3)", y_test, y_pred_dt_limited)

    print("CONCLUSIÓN FINAL")
    print("Recomiendo el modelo de Regresión Logística, ya que alcanzó el rendimiento más alto en el conjunto")
    print("de prueba con un Accuracy del 93.33%, y una precisión y sensibilidad (Recall) del 90.32%. Además, evita")
    print("el sobreajuste que presentó el árbol sin límite, permitiendo identificar a tiempo a la mayoría de los socios en riesgo.")

if __name__ == '__main__':
    main()